# Document Preprocessing for Dementia Simulation

This notebook processes uploaded PDFs and CSV files:
1. Cleans and extracts text from PDFs
2. Processes CSV data
3. Chunks text into passages
4. Saves processed data to `data/processed/`

In [ ]:
import os
import pandas as pd
import PyPDF2
import nltk
import json
import re
from pathlib import Path
from typing import List, Dict, Any
from tqdm import tqdm

# Download required NLTK data
try:
    nltk.data.find('tokenizers/punkt_tab')
except LookupError:
    nltk.download('punkt_tab')

try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords')

from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords

In [ ]:
# Configuration
UPLOAD_DIR = "data/uploads/"  # Directory containing uploaded files
PROCESSED_DIR = "data/processed/"
CHUNK_SIZE = 512  # Maximum tokens per chunk
OVERLAP = 50      # Overlap between chunks

# Ensure directories exist
os.makedirs(UPLOAD_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

print(f"Upload directory: {UPLOAD_DIR}")
print(f"Processed directory: {PROCESSED_DIR}")

In [ ]:
def clean_text(text: str) -> str:
    """Clean and normalize text content."""
    # Remove excessive whitespace
    text = re.sub(r'\s+', ' ', text)
    # Remove special characters but keep punctuation
    text = re.sub(r'[^a-zA-Z0-9\s.,;:!?\'\"\-()]', '', text)
    # Strip leading/trailing whitespace
    text = text.strip()
    return text

def process_pdf_file(pdf_path: str) -> str:
    """Extract text from PDF file."""
    text = ""
    try:
        with open(pdf_path, 'rb') as file:
            pdf_reader = PyPDF2.PdfReader(file)
            for page in pdf_reader.pages:
                page_text = page.extract_text()
                if page_text:
                    text += page_text + "\n"
    except Exception as e:
        print(f"Error reading PDF {pdf_path}: {e}")
        return ""
    
    return clean_text(text)

def process_csv_file(csv_path: str) -> str:
    """Process CSV file and extract text content."""
    try:
        df = pd.read_csv(csv_path)
        # Convert all columns to text and combine
        text_parts = []
        for col in df.columns:
            text_parts.append(f"{col}: {' '.join(df[col].astype(str).tolist())}")
        text = " ".join(text_parts)
        return clean_text(text)
    except Exception as e:
        print(f"Error reading CSV {csv_path}: {e}")
        return ""

In [ ]:
def chunk_text(text: str, chunk_size: int = CHUNK_SIZE, overlap: int = OVERLAP) -> List[str]:
    """Split text into overlapping chunks."""
    sentences = sent_tokenize(text)
    chunks = []
    current_chunk = ""
    current_length = 0
    
    for sentence in sentences:
        sentence_length = len(word_tokenize(sentence))
        
        if current_length + sentence_length > chunk_size and current_chunk:
            chunks.append(current_chunk.strip())
            
            # Create overlap by keeping the last few sentences
            overlap_sentences = sent_tokenize(current_chunk)
            overlap_text = " ".join(overlap_sentences[-2:]) if len(overlap_sentences) > 1 else ""
            overlap_length = len(word_tokenize(overlap_text))
            
            if overlap_length <= overlap:
                current_chunk = overlap_text + " " + sentence
                current_length = overlap_length + sentence_length
            else:
                current_chunk = sentence
                current_length = sentence_length
        else:
            current_chunk += " " + sentence if current_chunk else sentence
            current_length += sentence_length
    
    if current_chunk.strip():
        chunks.append(current_chunk.strip())
    
    return chunks

def process_document(file_path: str) -> Dict[str, Any]:
    """Process a single document and return metadata and chunks."""
    file_extension = Path(file_path).suffix.lower()
    file_name = Path(file_path).name
    
    if file_extension == '.pdf':
        text = process_pdf_file(file_path)
        file_type = 'pdf'
    elif file_extension == '.csv':
        text = process_csv_file(file_path)
        file_type = 'csv'
    else:
        print(f"Unsupported file type: {file_extension}")
        return None
    
    if not text.strip():
        print(f"No text extracted from {file_name}")
        return None
    
    chunks = chunk_text(text)
    
    return {
        'file_name': file_name,
        'file_type': file_type,
        'file_path': file_path,
        'text_length': len(text),
        'num_chunks': len(chunks),
        'chunks': chunks
    }

In [ ]:
# Process all uploaded files
def process_all_documents():
    """Process all documents in the upload directory."""
    processed_documents = []
    all_chunks = []
    
    # Find all PDF and CSV files in upload directory
    upload_path = Path(UPLOAD_DIR)
    pdf_files = list(upload_path.glob("*.pdf"))
    csv_files = list(upload_path.glob("*.csv"))
    
    all_files = pdf_files + csv_files
    
    if not all_files:
        print(f"No PDF or CSV files found in {UPLOAD_DIR}")
        return processed_documents, all_chunks
    
    print(f"Found {len(all_files)} files to process:")
    for file in all_files:
        print(f"  - {file.name}")
    
    # Process each document
    for file_path in tqdm(all_files, desc="Processing documents"):
        doc_data = process_document(str(file_path))
        
        if doc_data:
            processed_documents.append(doc_data)
            
            # Create chunk entries with metadata
            for i, chunk in enumerate(doc_data['chunks']):
                chunk_data = {
                    'id': f"{doc_data['file_name']}_chunk_{i}",
                    'text': chunk,
                    'source_file': doc_data['file_name'],
                    'file_type': doc_data['file_type'],
                    'chunk_index': i
                }
                all_chunks.append(chunk_data)
    
    return processed_documents, all_chunks

In [ ]:
# Run the preprocessing
print("Starting document preprocessing...")
print("=" * 50)

processed_documents, all_chunks = process_all_documents()

print("\n" + "=" * 50)
print("Processing Summary:")
print(f"- Processed {len(processed_documents)} documents")
print(f"- Generated {len(all_chunks)} text chunks")

In [ ]:
# Save results
if processed_documents and all_chunks:
    # Save document metadata
    metadata_file = os.path.join(PROCESSED_DIR, 'document_metadata.json')
    with open(metadata_file, 'w') as f:
        json.dump(processed_documents, f, indent=2)
    
    # Save text chunks as JSON
    chunks_file = os.path.join(PROCESSED_DIR, 'text_chunks.json')
    with open(chunks_file, 'w') as f:
        json.dump(all_chunks, f, indent=2)
    
    # Save chunks as CSV for easy viewing
    chunks_df = pd.DataFrame(all_chunks)
    chunks_csv = os.path.join(PROCESSED_DIR, 'text_chunks.csv')
    chunks_df.to_csv(chunks_csv, index=False)
    
    print("\nSaved processed data:")
    print(f"- Document metadata: {metadata_file}")
    print(f"- Text chunks (JSON): {chunks_file}")
    print(f"- Text chunks (CSV): {chunks_csv}")
    
    # Show sample chunks
    print("\nSample chunks:")
    for i, chunk in enumerate(all_chunks[:3]):
        print(f"\nChunk {i+1} (ID: {chunk['id']}):")
        print(f"Source: {chunk['source_file']}")
        print(f"Text: {chunk['text'][:200]}...")
else:
    print("\nNo documents were processed. Please add PDF or CSV files to the upload directory.")

## Next Steps

After preprocessing, run `build_index.py` to:
1. Load the processed chunks
2. Generate embeddings using sentence transformers
3. Create and save a FAISS index

```bash
python build_index.py
```